# Lab 10 - Escrow State Machine with Mutual Refund
Run the cells top to bottom (Runtime > Run all). Each `%%writefile` cell creates one file.

## File 1: `escrow.py` (the state machine)

In [ ]:
%%writefile escrow.py
"""Lab 10 - Escrow state machine (toy smart contract, simulated, no real blockchain)."""
from dataclasses import dataclass


@dataclass
class Escrow:
    buyer: str = 'Alice'
    seller: str = 'Bob'
    price: int = 150
    state: str = 'CREATED'
    escrowed: int = 0

    # CREATED -> FUNDED
    def deposit(self, actor, amount):
        if self.state != 'CREATED' or actor != self.buyer or amount != self.price:
            raise ValueError('Only buyer may deposit exact price before funding')
        self.escrowed = amount
        self.state = 'FUNDED'

    # FUNDED -> RELEASED
    def release(self, actor, delivered):
        if self.state != 'FUNDED' or actor != self.buyer or not delivered:
            raise ValueError('Release needs funded contract, buyer approval and delivery')
        self.state = 'RELEASED'
        return self.escrowed

    # FUNDED -> REFUNDED  (the required extension: needs BOTH parties)
    def refund(self, buyer_actor, seller_actor):
        if self.state != 'FUNDED' or buyer_actor != self.buyer or seller_actor != self.seller:
            raise ValueError('Refund requires FUNDED state and both parties consent')
        amount = self.escrowed
        self.escrowed = 0
        self.state = 'REFUNDED'
        return amount

    # CREATED -> CANCELLED
    def cancel(self, actor):
        if self.state != 'CREATED' or actor != self.buyer:
            raise ValueError('Only buyer can cancel before funding')
        self.state = 'CANCELLED'


## File 2: `run_demo.py` (reproduces the lab output)

In [ ]:
%%writefile run_demo.py
"""Lab 10 demo: reproduces the expected transcript from the lab sheet."""
from escrow import Escrow

# --- Scenario 1: normal order -------------------------------------------
e = Escrow()
for title, fn in [('Early release', lambda: e.release('Alice', True)),
                  ('Seller tries to fund', lambda: e.deposit('Bob', 150))]:
    try:
        fn()
    except ValueError as err:
        print('REJECTED', title, ':', err)

e.deposit('Alice', 150)
print('AFTER DEPOSIT:', e)

try:
    e.release('Alice', False)
except ValueError as err:
    print('REJECTED without delivery:', err)

paid = e.release('Alice', True)
print('AFTER RELEASE:', e, 'paid=', paid)

try:
    e.release('Alice', True)
except ValueError as err:
    print('REJECTED replay:', err)
assert paid == 150 and e.state == 'RELEASED'

# --- Scenario 2: mutual refund (required extension) ---------------------
refund_case = Escrow()
refund_case.deposit('Alice', 150)
try:
    refund_case.refund('Alice', 'Eve')
except ValueError as err:
    print('REJECTED unilateral refund:', err)

refund_amount = refund_case.refund('Alice', 'Bob')
print('AFTER MUTUALLY AGREED REFUND:', refund_case, 'refunded=', refund_amount)
assert refund_amount == 150 and refund_case.state == 'REFUNDED' and refund_case.escrowed == 0

try:
    refund_case.refund('Alice', 'Bob')
except ValueError as err:
    print('REJECTED repeat refund:', err)

print('PASS: state transitions and authorization checks enforced.')
print('NOTE: no real blockchain, funds, oracle, EVM or Solidity is involved.')


In [ ]:
!python run_demo.py

## File 3: `test_escrow.py` (unit tests for every transition)

In [ ]:
%%writefile test_escrow.py
"""Unit tests for every transition (including cancel, which the demo skips)."""
import unittest
from escrow import Escrow


class TestEscrow(unittest.TestCase):
    def setUp(self):
        self.e = Escrow()

    def test_deposit_ok(self):
        self.e.deposit('Alice', 150)
        self.assertEqual((self.e.state, self.e.escrowed), ('FUNDED', 150))

    def test_deposit_wrong_actor_or_amount(self):
        with self.assertRaises(ValueError): self.e.deposit('Bob', 150)
        with self.assertRaises(ValueError): self.e.deposit('Alice', 100)

    def test_early_release_rejected(self):
        with self.assertRaises(ValueError): self.e.release('Alice', True)

    def test_release_needs_delivery_and_buyer(self):
        self.e.deposit('Alice', 150)
        with self.assertRaises(ValueError): self.e.release('Alice', False)
        with self.assertRaises(ValueError): self.e.release('Bob', True)
        self.assertEqual(self.e.release('Alice', True), 150)
        self.assertEqual(self.e.state, 'RELEASED')

    def test_release_replay_rejected(self):
        self.e.deposit('Alice', 150); self.e.release('Alice', True)
        with self.assertRaises(ValueError): self.e.release('Alice', True)

    def test_refund_needs_both_parties(self):
        self.e.deposit('Alice', 150)
        with self.assertRaises(ValueError): self.e.refund('Alice', 'Eve')
        with self.assertRaises(ValueError): self.e.refund('Eve', 'Bob')
        self.assertEqual(self.e.refund('Alice', 'Bob'), 150)
        self.assertEqual((self.e.state, self.e.escrowed), ('REFUNDED', 0))

    def test_refund_before_funding_and_twice_rejected(self):
        with self.assertRaises(ValueError): self.e.refund('Alice', 'Bob')
        self.e.deposit('Alice', 150); self.e.refund('Alice', 'Bob')
        with self.assertRaises(ValueError): self.e.refund('Alice', 'Bob')

    def test_cancel(self):
        with self.assertRaises(ValueError): self.e.cancel('Bob')
        self.e.cancel('Alice')
        self.assertEqual(self.e.state, 'CANCELLED')
        with self.assertRaises(ValueError): self.e.deposit('Alice', 150)

    def test_cancel_after_funding_rejected(self):
        self.e.deposit('Alice', 150)
        with self.assertRaises(ValueError): self.e.cancel('Alice')


if __name__ == '__main__':
    unittest.main(verbosity=2)


In [ ]:
!python -m unittest -v test_escrow

In [ ]:
!python -m py_compile escrow.py run_demo.py && echo 'Compiles OK'